In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
%matplotlib inline

import math

from subprocess import check_output
print(check_output('dir .\\input', shell=True).decode('utf-8'))

 Volume in drive C has no label.
 Volume Serial Number is 2602-0473

 Directory of c:\Users\USER\Desktop\AI_study\kaggle_transcription\Natural language processing - classification, regression\2nd level. Mercari Price Suggestion Challenge\input

2026-09-19  오전 10:36    <DIR>          .
2026-09-19  오후 04:23    <DIR>          ..
2017-11-14  오전 09:43         9,595,930 sample_submission.csv
2026-03-07  오후 09:00           174,228 sample_submission.csv.7z
2026-03-07  오후 09:00         8,151,047 sample_submission_stg2.csv.zip
2017-11-11  오전 10:31       154,222,160 test.tsv
2026-03-07  오후 09:00        35,617,013 test.tsv.7z
2026-03-07  오후 09:00       308,669,128 test_stg2.tsv.zip
2017-11-11  오전 10:31       337,809,843 train.tsv
2026-03-07  오후 09:00        77,912,192 train.tsv.7z
               8 File(s)    932,151,541 bytes
               2 Dir(s)  293,111,009,280 bytes free



In [2]:
def rmsle(y, y_pred):
    assert len(y) == len(y_pred)
    to_sum = [(math.log(y_pred[i] + 1) - math.log(y[i] + 1)) ** 2.0 for i, pred in enumerate(y_pred)]
    return (sum(to_sum) * (1.0/len(y))) ** 0.5

In [3]:
print('Loading data...')
train = pd.read_table('./input/train.tsv')
test = pd.read_table('./input/test.tsv')
print(train.shape)
print(test.shape)

Loading data...
(1482535, 8)
(693359, 7)


In [4]:
print('Handling missing values...')
def handle_missing(dataset):
    dataset.category_name.fillna(value='missing', inplace=True)
    dataset.brand_name.fillna(value='missing', inplace=True)
    dataset.item_description.fillna(value='missing', inplace=True)
    return (dataset)

train = handle_missing(train)
test = handle_missing(test)
print(train.shape)
print(test.shape)

Handling missing values...


C:\Users\USER\AppData\Local\Temp\ipykernel_1480\2112688703.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataset.category_name.fillna(value='missing', inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_1480\2112688703.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.



(1482535, 8)
(693359, 7)


In [5]:
train.head(3)

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,missing,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...


In [6]:
print('Handling categorical variables')
le = LabelEncoder()

le.fit(np.hstack([train.category_name, test.category_name]))
train.category_name = le.transform(train.category_name)
test.category_name = le.transform(test.category_name)

le.fit(np.hstack([train.brand_name, test.brand_name]))
train.brand_name = le.transform(train.brand_name)
test.brand_name = le.transform(test.brand_name)
del le

train.head(3)

Handling categorical variables


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,829,5265,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,86,3889,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,1277,4588,10.0,1,Adorable top with a hint of lace and a key hol...


In [ ]:
print('Text to seq process...')
from tf_keras.preprocessing.text import Tokenizer
raw_text = np.hstack([train.item_description.str.lower(), train.name.str.lower()])

print('    Fitting tokenizer...')
tok_raw = Tokenizer()
tok_raw.fit_on_texts(raw_text)
print('    Transforming text to seq...')

train['seq_item_description'] = tok_raw.texts_to_sequences(train.item_description.str.lower())
test['seq_item_description'] = tok_raw.texts_to_sequences(test.item_description.str.lower())
train['seq_name'] = tok_raw.texts_to_sequences(train.name.str.lower())
test['seq_name'] = tok_raw.texts_to_sequences(test.name.str.lower())
train.head(3)

In [ ]:
max_name_seq = np.max([np.max(train.seq_name.apply(lambda x: len(x))), np.max(test.seq_name.apply(lambda x: len(x)))])
max_seq_item_description = np.max([np.max(train.seq_item_description.apply(lambda x: len(x))), np.max(test.seq_item_description.apply(lambda x: len(x)))])
print('max name seq ' + str(max_name_seq))
print('max item desc seq ' + str(max_seq_item_description))